# 📊 NB04：RAG 評估框架（RAGAS）

**目標：** 學習如何量化評估 RAG 系統的品質，建立可靠的評估流程。

**學習成果：**
- 理解為什麼評估 RAG 很困難（缺乏 Ground Truth）
- 掌握 RAGAS 四大核心指標：Context Recall、Context Precision、Faithfulness、Answer Relevancy
- 從零實作這些指標（不依賴外部框架，真正理解原理）
- 理解 LLM-as-Judge 的評估方法和偏見防範
- 了解生產系統的監控指標

> 💡 **一句話說重點：** 沒有評估，就沒有改進。RAGAS 讓你用 LLM 來評估 LLM，解決了「沒有標準答案」的難題。

## 1. 為什麼評估 RAG 很困難？

### 傳統 NLP 評估 vs RAG 評估

```
傳統 NLP（例如翻譯）：
  輸入：「The cat is on the mat」
  標準答案：「貓坐在墊子上」
  預測：「貓在墊子上面」
  → 可以算 BLEU、ROUGE 等自動指標

RAG 問答系統：
  輸入：「量子電腦如何影響加密技術？」
  「正確答案」是什麼？
  → 沒有單一正確答案！
  → 人工評估成本極高
  → 需要更聰明的評估方法
```

### RAGAS 的核心思路

**用 LLM 來評估 LLM！**

RAGAS（RAG Assessment）框架定義了四個不需要人工標準答案的評估指標：

```
┌─────────────────────────────────────────────────────┐
│ 評估 Retrieval（檢索）：                             │
│   • Context Recall：找到了多少相關資訊？             │
│   • Context Precision：找到的資訊有多精確？          │
│                                                     │
│ 評估 Generation（生成）：                           │
│   • Faithfulness：回答是否忠實於 Context？           │
│   • Answer Relevancy：回答是否切中問題？             │
└─────────────────────────────────────────────────────┘
```

## 2. 環境設定

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from dotenv import load_dotenv
from openai import OpenAI

matplotlib.rcParams['font.family'] = ['Arial Unicode MS', 'DejaVu Sans']

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✅ 環境設定完成")

## 3. 準備評估資料集

一個評估樣本（sample）包含四個元素：
- `question`：使用者問題
- `answer`：RAG 系統給出的回答
- `contexts`：RAG 系統檢索到的文件列表
- `ground_truth`：人工標注的參考答案（用於 Context Recall）

In [ ]:
# 評估資料集：模擬真實 RAG 系統的輸出
# 包含好的回答和壞的回答，用來驗證評估指標的效果

evaluation_samples = [
    {
        "id": "sample_01",
        "label": "好的 RAG 回答",
        "question": "BERT 是如何進行預訓練的？使用了哪些訓練任務？",
        "contexts": [
            "BERT（Bidirectional Encoder Representations from Transformers）是 Google 在 2018 年發布的預訓練語言模型。BERT 的預訓練任務包括遮罩語言模型（MLM）和下一句預測（NSP）。遮罩語言模型會隨機遮住 15% 的詞彙，讓模型根據上下文預測被遮住的詞。下一句預測任務讓模型判斷兩個句子是否在原文中相鄰。BERT 使用雙向 Transformer 編碼器，能夠同時考慮前後文。",
            "在 GLUE 基準測試上，BERT 創下了當時最佳的 80.5% 分數。BERT 的成功引發了預訓練語言模型的研究熱潮，後續出現了 RoBERTa、ALBERT 等改進版本。"
        ],
        "answer": "BERT 使用兩種預訓練任務：(1) 遮罩語言模型（MLM）- 隨機遮住 15% 的詞彙，讓模型根據前後文預測被遮住的詞；(2) 下一句預測（NSP）- 讓模型判斷兩個句子是否在原文中相鄰。BERT 使用雙向 Transformer 架構，能同時考慮前後文，這是它與 GPT 的主要差異。",
        "ground_truth": "BERT 的預訓練使用了兩個任務：遮罩語言模型（MLM）和下一句預測（NSP）。MLM 會遮住 15% 的詞彙要求模型預測，NSP 讓模型學習句子間的關係。"
    },
    {
        "id": "sample_02",
        "label": "幻覺問題（Hallucination）",
        "question": "GPT-4 的參數量是多少？",
        "contexts": [
            "GPT-4 是 OpenAI 在 2023 年 3 月發布的大型語言模型，參數量估計超過 1 兆。GPT-4 在律師資格考試中達到前 10% 的成績。GPT-4 Turbo 版本具有 128K token 上下文窗口。"
        ],
        "answer": "根據公開資訊，GPT-4 的確切參數量為 170 兆（170 trillion）個參數。這是 OpenAI 在 2023 年官方公佈的數字，使其成為目前最大的語言模型之一。",
        "ground_truth": "GPT-4 的確切參數量 OpenAI 未公開，估計超過 1 兆，但 170 兆的說法未經證實。"
    },
    {
        "id": "sample_03",
        "label": "Context 不相關",
        "question": "向量資料庫的 HNSW 索引是什麼？",
        "contexts": [
            "RAG 系統能有效減少 LLM 的幻覺問題，並支援知識即時更新。RAG 包含 Retriever 和 Generator 兩個核心元件。",
            "Embedding 模型將文字轉換為高維稠密向量，用於語意相似度計算。OpenAI 的 text-embedding-3-small 輸出 1536 維向量。"
        ],
        "answer": "我在提供的資料中找不到關於 HNSW 索引的具體說明。根據我的知識，HNSW（Hierarchical Navigable Small World）是一種用於近似最近鄰搜索的圖結構索引，常用於向量資料庫中以加速相似度搜索。",
        "ground_truth": "HNSW（分層可導航小世界圖）是向量資料庫中常用的近似最近鄰搜索索引，適合高速搜索場景。"
    },
    {
        "id": "sample_04",
        "label": "回答不切題",
        "question": "RAG 和 Fine-tuning 各自適合什麼場景？",
        "contexts": [
            "RAG 架構結合信息檢索和文字生成，能有效減少幻覺並支援知識即時更新。Fine-tuning 是將預訓練模型針對特定任務進行進一步訓練。全參數微調更新所有參數但成本高，LoRA 只更新少量參數降低成本。"
        ],
        "answer": "RAG 和 Fine-tuning 都是非常重要的技術。它們各有優缺點，在深度學習領域都有廣泛應用。選擇哪種方法取決於你的具體需求、資源和場景。總的來說，兩者都可以提升語言模型的表現。我建議你深入研究這兩種技術並根據實際情況做出選擇。",
        "ground_truth": "RAG 適合知識需要頻繁更新、需要引用來源的場景；Fine-tuning 適合需要改變模型行為風格、輸出特定格式的場景。"
    }
]

print(f"已準備 {len(evaluation_samples)} 個評估樣本：")
for s in evaluation_samples:
    print(f"  [{s['id']}] {s['label']}")

## 4. 指標一：Faithfulness（忠實度）

**定義：** 回答中的每個陳述是否都能從 Context 中找到依據？

$$\text{Faithfulness} = \frac{\text{Context 中有依據的陳述數量}}{\text{回答中的總陳述數量}}$$

**作用：** 偵測幻覺（Hallucination）— 模型編造了 Context 中沒有的資訊

```
高 Faithfulness (≈1.0)：回答完全基於 Context
低 Faithfulness (≈0.0)：回答大量包含 Context 沒有的內容
```

In [ ]:
def evaluate_faithfulness(question: str, answer: str, contexts: list[str]) -> dict:
    """
    評估回答的忠實度（Faithfulness）
    步驟：
    1. 從回答中提取所有陳述
    2. 逐一判斷每個陳述是否有 Context 依據
    3. 計算有依據的比例
    """
    context_combined = "\n\n".join(contexts)
    
    # Step 1: 提取回答中的所有陳述
    extract_prompt = f"""請從以下回答中提取所有獨立的事實陳述。
每行輸出一個陳述，用數字編號。
只提取明確的事實性陳述，不要包含問句或模糊表達。

回答：{answer}

格式：
1. [陳述一]
2. [陳述二]
..."""

    extract_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": extract_prompt}],
        temperature=0
    )
    statements_text = extract_response.choices[0].message.content
    
    # 解析陳述列表
    statements = []
    for line in statements_text.strip().split("\n"):
        line = line.strip()
        if line and line[0].isdigit():
            # 移除編號
            statement = line.split(".", 1)[-1].strip().strip("[]")
            if statement:
                statements.append(statement)
    
    # Step 2: 判斷每個陳述是否有 Context 依據
    verdicts = []
    for stmt in statements:
        verify_prompt = f"""根據以下參考資料，判斷這個陳述是否有依據。

參考資料：
{context_combined}

陳述：{stmt}

請判斷：這個陳述是否可以從參考資料中找到依據或推理出來？
只回答 YES 或 NO。"""
        
        verdict_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": verify_prompt}],
            temperature=0
        )
        verdict = verdict_response.choices[0].message.content.strip().upper()
        verdicts.append("YES" in verdict)
    
    # Step 3: 計算分數
    supported_count = sum(verdicts)
    total_count = len(statements)
    score = supported_count / total_count if total_count > 0 else 0
    
    return {
        "score": score,
        "supported": supported_count,
        "total": total_count,
        "statements": statements,
        "verdicts": verdicts
    }


# 測試：比較「好的回答」和「有幻覺的回答」
print("=" * 60)
print("Faithfulness 評估")
print("=" * 60)

for sample in evaluation_samples[:2]:
    print(f"\n[{sample['label']}]")
    print(f"問題：{sample['question']}")
    
    result = evaluate_faithfulness(
        sample["question"],
        sample["answer"],
        sample["contexts"]
    )
    
    print(f"\nFaithfulness 分數：{result['score']:.2f} ({result['supported']}/{result['total']} 個陳述有依據)")
    print("陳述分析：")
    for stmt, verdict in zip(result["statements"], result["verdicts"]):
        icon = "✅" if verdict else "❌"
        print(f"  {icon} {stmt}")
    print("-" * 50)

## 5. 指標二：Answer Relevancy（回答相關性）

**定義：** 回答是否切中問題？回答有沒有答非所問或廢話連篇？

**方法（反向生成法）：**
1. 根據回答反向生成多個「可能的問題」
2. 計算這些反向生成問題與原始問題的相似度
3. 如果回答真的回答了問題，反向生成的問題應該和原始問題很相似

$$\text{Answer Relevancy} = \frac{1}{N} \sum_{i=1}^{N} \text{cos\_sim}(q_i, q_{\text{original}})$$

In [ ]:
def cosine_similarity(v1: list, v2: list) -> float:
    v1, v2 = np.array(v1), np.array(v2)
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))


def evaluate_answer_relevancy(question: str, answer: str, n_questions: int = 3) -> dict:
    """
    評估回答相關性（Answer Relevancy）
    用「反向生成問題」方法評估回答是否切中問題
    """
    # Step 1: 根據回答生成多個「可能的問題」
    generate_prompt = f"""根據以下回答，生成 {n_questions} 個可能導致此回答的不同問題。
這些問題應該是原始問題的可能變形，保留核心意圖。

回答：{answer}

請輸出 {n_questions} 個問題，每行一個，不加編號："""
    
    gen_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": generate_prompt}],
        temperature=0.5
    )
    
    generated_questions = [
        q.strip() for q in gen_response.choices[0].message.content.strip().split("\n")
        if q.strip()
    ][:n_questions]
    
    # Step 2: 計算反向問題和原始問題的相似度
    all_questions = [question] + generated_questions
    
    emb_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=all_questions
    )
    
    original_embedding = emb_response.data[0].embedding
    generated_embeddings = [emb_response.data[i+1].embedding for i in range(len(generated_questions))]
    
    similarities = [
        cosine_similarity(original_embedding, gen_emb)
        for gen_emb in generated_embeddings
    ]
    
    score = np.mean(similarities)
    
    return {
        "score": float(score),
        "generated_questions": generated_questions,
        "similarities": similarities
    }


# 測試：比較切題的回答和不切題的回答
print("=" * 60)
print("Answer Relevancy 評估")
print("=" * 60)

# 測試樣本 1（好的回答）和樣本 4（不切題的回答）
for idx in [0, 3]:
    sample = evaluation_samples[idx]
    print(f"\n[{sample['label']}]")
    print(f"原始問題：{sample['question']}")
    print(f"回答（前 100 字）：{sample['answer'][:100]}...")
    
    result = evaluate_answer_relevancy(sample["question"], sample["answer"])
    
    print(f"\nAnswer Relevancy 分數：{result['score']:.4f}")
    print("反向生成的問題：")
    for q, sim in zip(result["generated_questions"], result["similarities"]):
        print(f"  (相似度 {sim:.4f}) {q}")
    print("-" * 50)

## 6. 指標三：Context Precision（Context 精確度）

**定義：** 檢索到的文件中，有多少是真正有用的？（避免引入雜訊）

$$\text{Context Precision} = \frac{\text{有用的 Context 數量}}{\text{總 Context 數量}}$$

**作用：** 檢查 RAG 是否帶入太多不相關的 Context（降低生成品質，增加 token 成本）

In [ ]:
def evaluate_context_precision(question: str, answer: str, contexts: list[str]) -> dict:
    """
    評估 Context 精確度（Context Precision）
    判斷每個 Context 是否對生成正確回答有幫助
    """
    verdicts = []
    
    for i, context in enumerate(contexts):
        prompt = f"""請判斷以下「參考文件」是否對回答這個「問題」有幫助或相關。

問題：{question}

參考文件：{context}

回答：{answer}

這份參考文件對回答這個問題是否有幫助或包含相關資訊？
只回答 YES 或 NO。"""
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        
        verdict = "YES" in response.choices[0].message.content.strip().upper()
        verdicts.append(verdict)
    
    score = sum(verdicts) / len(verdicts) if verdicts else 0
    
    return {
        "score": score,
        "relevant_count": sum(verdicts),
        "total_count": len(verdicts),
        "verdicts": verdicts
    }


# 測試：Context 不相關的樣本 vs 相關的樣本
print("=" * 60)
print("Context Precision 評估")
print("=" * 60)

# 樣本 1（好的 Context）和樣本 3（不相關的 Context）
for idx in [0, 2]:
    sample = evaluation_samples[idx]
    print(f"\n[{sample['label']}]")
    print(f"問題：{sample['question']}")
    print(f"Context 數量：{len(sample['contexts'])}")
    
    result = evaluate_context_precision(
        sample["question"],
        sample["answer"],
        sample["contexts"]
    )
    
    print(f"\nContext Precision 分數：{result['score']:.2f} ({result['relevant_count']}/{result['total_count']} 個 Context 相關)")
    for i, (ctx, verdict) in enumerate(zip(sample["contexts"], result["verdicts"])):
        icon = "✅" if verdict else "❌"
        print(f"  {icon} Context {i+1}: {ctx[:80]}...")
    print("-" * 50)

## 7. 指標四：Context Recall（Context 召回率）

**定義：** 回答問題所需的資訊，有多少比例被正確檢索出來？

**需要：** Ground Truth（參考答案）來判斷什麼資訊是「需要的」

$$\text{Context Recall} = \frac{\text{Ground Truth 中被 Context 覆蓋的資訊量}}{\text{Ground Truth 的總資訊量}}$$

In [ ]:
def evaluate_context_recall(question: str, contexts: list[str], ground_truth: str) -> dict:
    """
    評估 Context 召回率（Context Recall）
    判斷 Ground Truth 中的每個陳述是否能從 Context 中找到
    """
    context_combined = "\n\n".join(contexts)
    
    # Step 1: 從 Ground Truth 提取所有陳述
    extract_prompt = f"""請從以下「參考答案」中提取所有獨立的事實陳述。
每行一個陳述，用數字編號。

參考答案：{ground_truth}

格式：
1. [陳述一]
2. [陳述二]
..."""
    
    extract_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": extract_prompt}],
        temperature=0
    )
    
    statements = []
    for line in extract_response.choices[0].message.content.strip().split("\n"):
        line = line.strip()
        if line and line[0].isdigit():
            statement = line.split(".", 1)[-1].strip().strip("[]")
            if statement:
                statements.append(statement)
    
    # Step 2: 判斷每個陳述是否能從 Context 中找到
    verdicts = []
    for stmt in statements:
        check_prompt = f"""根據以下「參考資料」，判斷這個「陳述」是否可以從中找到支持或推理出來。

參考資料：
{context_combined}

陳述：{stmt}

只回答 YES 或 NO。"""
        
        check_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": check_prompt}],
            temperature=0
        )
        verdict = "YES" in check_response.choices[0].message.content.strip().upper()
        verdicts.append(verdict)
    
    covered_count = sum(verdicts)
    total_count = len(statements)
    score = covered_count / total_count if total_count > 0 else 0
    
    return {
        "score": score,
        "covered": covered_count,
        "total": total_count,
        "statements": statements,
        "verdicts": verdicts
    }


# 測試
print("=" * 60)
print("Context Recall 評估")
print("=" * 60)

for idx in [0, 2]:
    sample = evaluation_samples[idx]
    print(f"\n[{sample['label']}]")
    print(f"問題：{sample['question']}")
    print(f"參考答案：{sample['ground_truth']}")
    
    result = evaluate_context_recall(
        sample["question"],
        sample["contexts"],
        sample["ground_truth"]
    )
    
    print(f"\nContext Recall 分數：{result['score']:.2f} ({result['covered']}/{result['total']} 個陳述被 Context 覆蓋)")
    for stmt, verdict in zip(result["statements"], result["verdicts"]):
        icon = "✅" if verdict else "❌"
        print(f"  {icon} {stmt}")
    print("-" * 50)

## 8. 完整評估 Pipeline

整合四個指標，對整個評估集打分並視覺化結果。

In [ ]:
def full_ragas_evaluation(samples: list[dict]) -> list[dict]:
    """對所有樣本執行完整的 RAGAS 評估"""
    results = []
    
    for i, sample in enumerate(samples):
        print(f"\n評估 [{i+1}/{len(samples)}] {sample['label']}...")
        
        # 評估四個指標
        faithfulness = evaluate_faithfulness(
            sample["question"], sample["answer"], sample["contexts"]
        )
        print(f"  Faithfulness: {faithfulness['score']:.3f}")
        
        answer_relevancy = evaluate_answer_relevancy(
            sample["question"], sample["answer"]
        )
        print(f"  Answer Relevancy: {answer_relevancy['score']:.3f}")
        
        context_precision = evaluate_context_precision(
            sample["question"], sample["answer"], sample["contexts"]
        )
        print(f"  Context Precision: {context_precision['score']:.3f}")
        
        context_recall = evaluate_context_recall(
            sample["question"], sample["contexts"], sample["ground_truth"]
        )
        print(f"  Context Recall: {context_recall['score']:.3f}")
        
        results.append({
            "id": sample["id"],
            "label": sample["label"],
            "faithfulness": faithfulness["score"],
            "answer_relevancy": answer_relevancy["score"],
            "context_precision": context_precision["score"],
            "context_recall": context_recall["score"],
        })
    
    return results


print("開始完整 RAGAS 評估...")
print("=" * 60)
eval_results = full_ragas_evaluation(evaluation_samples)
print("\n✅ 評估完成！")

In [ ]:
# 整理結果表格
print("\n" + "=" * 80)
print(f"{'樣本':<20} {'Faithfulness':>15} {'Ans Relevancy':>15} {'Ctx Precision':>15} {'Ctx Recall':>12}")
print("-" * 80)

for r in eval_results:
    label = r['label'][:18]
    print(f"{label:<20} {r['faithfulness']:>15.3f} {r['answer_relevancy']:>15.3f} {r['context_precision']:>15.3f} {r['context_recall']:>12.3f}")

print("-" * 80)
avg_faith = np.mean([r['faithfulness'] for r in eval_results])
avg_ans = np.mean([r['answer_relevancy'] for r in eval_results])
avg_prec = np.mean([r['context_precision'] for r in eval_results])
avg_rec = np.mean([r['context_recall'] for r in eval_results])
print(f"{'平均':<20} {avg_faith:>15.3f} {avg_ans:>15.3f} {avg_prec:>15.3f} {avg_rec:>12.3f}")

In [ ]:
# 視覺化：雷達圖比較各樣本的 RAGAS 分數
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 圖1：各樣本的指標條形圖
ax1 = axes[0]
metrics = ['Faithfulness', 'Ans\nRelevancy', 'Ctx\nPrecision', 'Ctx\nRecall']
x = np.arange(len(metrics))
width = 0.2

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
for i, (r, color) in enumerate(zip(eval_results, colors)):
    values = [r['faithfulness'], r['answer_relevancy'], r['context_precision'], r['context_recall']]
    bars = ax1.bar(x + i*width, values, width, label=r['label'][:15], color=color, alpha=0.8)

ax1.set_xlabel('RAGAS 指標')
ax1.set_ylabel('分數 (0-1)')
ax1.set_title('各樣本的 RAGAS 四大指標分數')
ax1.set_xticks(x + width * 1.5)
ax1.set_xticklabels(metrics)
ax1.set_ylim(0, 1.1)
ax1.legend(loc='upper right', fontsize=7)
ax1.axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='最低合格線')
ax1.grid(axis='y', alpha=0.3)

# 圖2：平均分數
ax2 = axes[1]
avg_scores = [avg_faith, avg_ans, avg_prec, avg_rec]
bar_colors = ['#66BB6A' if s >= 0.7 else '#EF5350' for s in avg_scores]

bars = ax2.bar(metrics, avg_scores, color=bar_colors, alpha=0.8)
ax2.set_ylabel('平均分數 (0-1)')
ax2.set_title('整體平均 RAGAS 分數')
ax2.set_ylim(0, 1.1)
ax2.axhline(y=0.7, color='red', linestyle='--', alpha=0.7, label='最低合格線 (0.7)')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

for bar, score in zip(bars, avg_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{score:.2f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('ragas_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print("圖表已儲存為 ragas_evaluation.png")

## 9. LLM-as-Judge 的偏見與防範

使用 LLM 評估 LLM 有幾個已知偏見需要注意：

| 偏見類型 | 說明 | 防範方法 |
|---------|------|----------|
| **位置偏見** | 評估多個選項時，偏好第一個或最後一個 | 交換順序多次評估取平均 |
| **冗長偏見** | 傾向給較長的回答更高分 | 明確指定評估標準 |
| **自我偏見** | GPT-4 傾向認為 GPT 的回答更好 | 用不同廠商的 LLM 交叉評估 |
| **格式偏見** | 偏好有格式（bullet points、粗體）的回答 | 在 prompt 中說明不考慮格式 |

In [ ]:
def robust_llm_judge(
    question: str,
    answer_a: str,
    answer_b: str,
    n_trials: int = 2
) -> dict:
    """
    更穩健的 LLM-as-Judge 評估
    通過交換順序來抵消位置偏見
    """
    results = []
    
    def judge(q, a1, a2, label_1, label_2) -> str:
        prompt = f"""請根據以下標準評估哪個回答更好：
1. 準確性：回答是否正確
2. 相關性：回答是否切中問題
3. 完整性：是否涵蓋問題的主要面向

問題：{q}

回答 A：
{a1}

回答 B：
{a2}

哪個回答更好？只回答 A 或 B 或 EQUAL。不要解釋原因。"""
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        verdict = response.choices[0].message.content.strip().upper()
        
        if "A" in verdict and "B" not in verdict:
            return label_1
        elif "B" in verdict and "A" not in verdict:
            return label_2
        else:
            return "EQUAL"
    
    # 正序評估
    result1 = judge(question, answer_a, answer_b, "A", "B")
    results.append(result1)
    
    # 逆序評估（交換 A 和 B）
    result2 = judge(question, answer_b, answer_a, "B", "A")
    results.append(result2)
    
    # 統計
    a_wins = results.count("A")
    b_wins = results.count("B")
    
    if a_wins > b_wins:
        winner = "A"
    elif b_wins > a_wins:
        winner = "B"
    else:
        winner = "EQUAL"
    
    return {
        "winner": winner,
        "a_wins": a_wins,
        "b_wins": b_wins,
        "individual_results": results
    }


# 示範：比較好的回答 vs 幻覺回答
question = "BERT 使用什麼預訓練任務？"
good_answer = "BERT 使用兩種預訓練任務：(1) 遮罩語言模型（MLM）- 隨機遮住 15% 的詞彙讓模型預測；(2) 下一句預測（NSP）- 讓模型判斷兩句話是否在原文相鄰。"
bad_answer = "BERT 使用監督學習方式訓練，包含情感分析、命名實體識別和機器翻譯三個任務，在每個任務上分別訓練後再合併模型參數。"

print("LLM-as-Judge 比較評估（交換順序防止位置偏見）")
print("=" * 60)
print(f"問題：{question}")
print(f"\n回答 A：{good_answer}")
print(f"\n回答 B：{bad_answer}")

judge_result = robust_llm_judge(question, good_answer, bad_answer)
print(f"\n判決結果：{judge_result['winner']} 更好")
print(f"正序評估：{judge_result['individual_results'][0]}")
print(f"逆序評估：{judge_result['individual_results'][1]}")
print(f"最終：A 勝 {judge_result['a_wins']} 次，B 勝 {judge_result['b_wins']} 次")

## 10. 生產系統監控指標

除了 RAGAS 的離線評估，生產系統還需要監控以下線上指標：

In [ ]:
# 模擬生產系統監控數據
import random
import datetime

random.seed(42)

# 模擬一週的查詢日誌
days = 7
queries_per_day = 100

monitoring_data = []
for day in range(days):
    for _ in range(queries_per_day):
        monitoring_data.append({
            "day": day,
            "retrieval_latency_ms": random.gauss(80, 20),     # 檢索延遲
            "generation_latency_ms": random.gauss(1500, 300), # 生成延遲
            "total_tokens": random.randint(500, 2000),         # Token 使用量
            "user_thumbs_up": random.random() > 0.25,          # 用戶正面回饋
            "fallback_triggered": random.random() > 0.9,       # 是否觸發 fallback
            "n_contexts_retrieved": random.choice([3, 3, 3, 5, 5]),
        })

# 計算關鍵指標
latencies = [d["retrieval_latency_ms"] + d["generation_latency_ms"] for d in monitoring_data]
p50 = np.percentile(latencies, 50)
p95 = np.percentile(latencies, 95)
p99 = np.percentile(latencies, 99)

thumbs_up_rate = sum(d["user_thumbs_up"] for d in monitoring_data) / len(monitoring_data)
fallback_rate = sum(d["fallback_triggered"] for d in monitoring_data) / len(monitoring_data)
avg_tokens = np.mean([d["total_tokens"] for d in monitoring_data])

print("=" * 60)
print("生產系統監控報告（過去 7 天）")
print("=" * 60)
print(f"\n📈 延遲指標：")
print(f"  P50 延遲：{p50:.0f} ms")
print(f"  P95 延遲：{p95:.0f} ms  {'✅' if p95 < 3000 else '⚠️ 超標！'}")
print(f"  P99 延遲：{p99:.0f} ms  {'✅' if p99 < 5000 else '⚠️ 超標！'}")
print(f"\n👍 品質指標：")
print(f"  用戶好評率：{thumbs_up_rate:.1%}  {'✅' if thumbs_up_rate > 0.7 else '⚠️ 需改善！'}")
print(f"  Fallback 率：{fallback_rate:.1%}  {'✅' if fallback_rate < 0.1 else '⚠️ 過高！'}")
print(f"\n💰 成本指標：")
print(f"  平均每次查詢 Token 數：{avg_tokens:.0f} tokens")
estimated_daily_cost = avg_tokens * queries_per_day * 0.0002 / 1000  # 估算成本
print(f"  每日估算成本：${estimated_daily_cost:.2f}")

In [ ]:
# 視覺化監控趨勢
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 按天計算指標
daily_p95 = []
daily_thumbs = []
daily_tokens = []
daily_fallback = []

for day in range(days):
    day_data = [d for d in monitoring_data if d["day"] == day]
    day_latencies = [d["retrieval_latency_ms"] + d["generation_latency_ms"] for d in day_data]
    daily_p95.append(np.percentile(day_latencies, 95))
    daily_thumbs.append(sum(d["user_thumbs_up"] for d in day_data) / len(day_data))
    daily_tokens.append(np.mean([d["total_tokens"] for d in day_data]))
    daily_fallback.append(sum(d["fallback_triggered"] for d in day_data) / len(day_data))

day_labels = [f"Day {i+1}" for i in range(days)]

# 延遲趨勢
axes[0, 0].plot(day_labels, daily_p95, marker='o', color='#2196F3')
axes[0, 0].axhline(y=3000, color='red', linestyle='--', alpha=0.7, label='P95 目標 (3s)')
axes[0, 0].set_title('P95 延遲趨勢 (ms)')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 好評率趨勢
axes[0, 1].plot(day_labels, [t*100 for t in daily_thumbs], marker='s', color='#4CAF50')
axes[0, 1].axhline(y=70, color='red', linestyle='--', alpha=0.7, label='目標 (70%)')
axes[0, 1].set_title('用戶好評率 (%)')
axes[0, 1].set_ylim(0, 100)
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Token 使用趨勢
axes[1, 0].bar(day_labels, daily_tokens, color='#FF9800', alpha=0.8)
axes[1, 0].set_title('平均 Token 使用量')
axes[1, 0].grid(axis='y', alpha=0.3)

# Fallback 率
fallback_colors = ['#EF5350' if f > 0.1 else '#66BB6A' for f in daily_fallback]
axes[1, 1].bar(day_labels, [f*100 for f in daily_fallback], color=fallback_colors, alpha=0.8)
axes[1, 1].axhline(y=10, color='red', linestyle='--', alpha=0.7, label='上限 (10%)')
axes[1, 1].set_title('Fallback 觸發率 (%)')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.suptitle('RAG 生產系統監控儀表板', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('rag_monitoring.png', dpi=150, bbox_inches='tight')
plt.show()
print("圖表已儲存為 rag_monitoring.png")

## 11. 完整評估框架總結

```
RAG 評估體系
├── 離線評估（開發/測試階段）
│   ├── 檢索評估
│   │   ├── Context Recall：找到必要資訊的比例
│   │   ├── Context Precision：找到的資訊有多精確
│   │   ├── MRR（平均倒數排名）
│   │   └── Hit Rate@K
│   └── 生成評估
│       ├── Faithfulness：忠實於 Context
│       ├── Answer Relevancy：切中問題
│       ├── Answer Correctness：與 Ground Truth 比較
│       └── LLM-as-Judge（位置/順序交換防偏）
└── 線上監控（生產階段）
    ├── 延遲：P50/P95/P99
    ├── 用戶回饋：好評率、差評率
    ├── 成本：Token 使用量、API 費用
    └── 可靠性：Fallback 率、錯誤率
```

### 面試關鍵問答
> **面試官：** 你如何評估一個 RAG 系統的好壞？
>
> **好答案：** 「我會分兩層評估。第一層是組件評估：用 Context Recall 和 Context Precision 分別評估檢索器；用 Faithfulness 檢測幻覺，用 Answer Relevancy 看回答是否切題。第二層是端對端評估：建立黃金測試集，用 LLM-as-Judge 比較不同版本，並交換順序評估多次以抵消位置偏見。在生產環境中，我還會持續監控 P95 延遲、用戶好評率和 Fallback 率作為實時品質指標。」

## 小結

1. ✅ **RAG 評估的難點**：沒有單一正確答案，需要用 LLM 評估 LLM
2. ✅ **RAGAS 四大指標**：
   - **Faithfulness**：回答是否忠實於 Context（偵測幻覺）
   - **Answer Relevancy**：回答是否切中問題（反向生成法）
   - **Context Precision**：檢索的 Context 有多精確
   - **Context Recall**：必要資訊被找到了多少
3. ✅ **LLM-as-Judge 偏見防範**：交換順序、多次採樣
4. ✅ **生產監控指標**：延遲、好評率、成本、Fallback 率

---

## 🎓 整個系列回顧

| NB | 主題 | 核心技能 |
|----|------|----------|
| **NB01** | RAG 基礎概念 | 完整 Pipeline、ChromaDB + OpenAI |
| **NB02** | Chunking 策略 | 固定/重疊/遞迴/語意切塊比較 |
| **NB03** | 進階 RAG | 混合搜索（BM25+向量）、RRF、Reranking |
| **NB04** | 評估框架 | RAGAS 四指標、LLM-as-Judge、生產監控 |

**恭喜完成 RAG 學習系列！🎉**